# 面试题：怎样验证工具结果而不是相信工具文本？

回答要点：工具文本说成功不等于事实。对写操作要验证响应 schema、关联请求 ID、权威资源状态、版本变化和业务后置条件；超时应判为 unknown 并回读，而不是盲目重试。最终状态必须来自权威系统，模型只能解释结果。

## 真实案例

订单取消工具返回六条事件：有的文本成功但版本没变，有的超时后权威状态已取消，有的关联 ID 错误。

## 基线

基线只要响应文本包含“成功”就向用户宣称完成。

## 结果解读

手写 verifier 同时核验 correlation id、状态、目标订单和版本。

## 失败案例

超时未知不能直接重试，否则可能对已完成的写操作造成重复副作用。

In [1]:
events = [{'id':'E01','request':'r1','reply':'取消成功','reply_request':'r1','state':'cancelled','expected':'cancelled','version_before':3,'version_after':4}, {'id':'E02','request':'r2','reply':'取消成功','reply_request':'r2','state':'paid','expected':'cancelled','version_before':5,'version_after':5}, {'id':'E03','request':'r3','reply':'超时','reply_request':'r3','state':'cancelled','expected':'cancelled','version_before':1,'version_after':2}, {'id':'E04','request':'r4','reply':'取消成功','reply_request':'other','state':'cancelled','expected':'cancelled','version_before':2,'version_after':3}, {'id':'E05','request':'r5','reply':'取消失败','reply_request':'r5','state':'paid','expected':'cancelled','version_before':1,'version_after':1}, {'id':'E06','request':'r6','reply':'取消成功','reply_request':'r6','state':'cancelled','expected':'cancelled','version_before':8,'version_after':9}]  # 构造六个具有响应文本和权威状态的取消事件。
print('工具事件输入:', events)  # 输出包含响应、关联 ID 和版本的原始账本。
print('教学说明：state/version 代表订单服务的权威回读，不是模型生成文本。')  # 强调可信来源边界。

工具事件输入: [{'id': 'E01', 'request': 'r1', 'reply': '取消成功', 'reply_request': 'r1', 'state': 'cancelled', 'expected': 'cancelled', 'version_before': 3, 'version_after': 4}, {'id': 'E02', 'request': 'r2', 'reply': '取消成功', 'reply_request': 'r2', 'state': 'paid', 'expected': 'cancelled', 'version_before': 5, 'version_after': 5}, {'id': 'E03', 'request': 'r3', 'reply': '超时', 'reply_request': 'r3', 'state': 'cancelled', 'expected': 'cancelled', 'version_before': 1, 'version_after': 2}, {'id': 'E04', 'request': 'r4', 'reply': '取消成功', 'reply_request': 'other', 'state': 'cancelled', 'expected': 'cancelled', 'version_before': 2, 'version_after': 3}, {'id': 'E05', 'request': 'r5', 'reply': '取消失败', 'reply_request': 'r5', 'state': 'paid', 'expected': 'cancelled', 'version_before': 1, 'version_after': 1}, {'id': 'E06', 'request': 'r6', 'reply': '取消成功', 'reply_request': 'r6', 'state': 'cancelled', 'expected': 'cancelled', 'version_before': 8, 'version_after': 9}]
教学说明：state/version 代表订单服务的权威回读，不是模型生成文本。

In [2]:
def trust_text(row):  # 定义只信任自然语言响应的错误基线。
    return '完成' if '成功' in row['reply'] else '失败'  # 把文本关键词直接映射成面向用户的状态。
baseline = [(row['id'], trust_text(row)) for row in events]  # 对六条事件运行文本信任基线。
print('文本基线:', baseline)  # 输出会把 E02 和 E04 误报成功的结果。

文本基线: [('E01', '完成'), ('E02', '完成'), ('E03', '失败'), ('E04', '完成'), ('E05', '失败'), ('E06', '完成')]


In [3]:
def verify(row):  # 定义基于权威状态的写操作后置验证。
    correlation_ok = row['request'] == row['reply_request']  # 检查工具响应是否属于当前调用。
    state_ok = row['state'] == row['expected']  # 检查订单服务的最终状态是否满足目标。
    version_ok = row['version_after'] > row['version_before']  # 检查写操作是否确实改变了资源版本。
    if correlation_ok and state_ok and version_ok and row['reply'] != '超时':  # 非超时且全部后置条件成立才确认完成。
        return 'verified', {'correlation':correlation_ok,'state':state_ok,'version':version_ok}  # 返回可审计的成功证据。
    if row['reply'] == '超时' and state_ok:  # 特判超时但权威状态已达成的情形。
        return 'verified_after_readback', {'correlation':correlation_ok,'state':state_ok,'version':version_ok}  # 说明无需盲目重试。
    return 'unknown_or_failed', {'correlation':correlation_ok,'state':state_ok,'version':version_ok}  # 将其余情况留给回读、重试或人工升级。

In [4]:
results = [(row['id'],) + verify(row) for row in events]  # 对六条工具事件执行权威后置验证。
print('id | 验证结论 | 证据')  # 输出验证结果表标题。
for item in results:  # 遍历每条事件的结论与中间证据。
    print(item[0], item[1], item[2])  # 输出关联、状态和版本三个验证量。
print('文本误报数:', sum(trust_text(row) == '完成' and verify(row)[0] == 'unknown_or_failed' for row in events))  # 统计盲信文本造成的错误完成。

id | 验证结论 | 证据
E01 verified {'correlation': True, 'state': True, 'version': True}
E02 unknown_or_failed {'correlation': True, 'state': False, 'version': False}
E03 verified_after_readback {'correlation': True, 'state': True, 'version': True}
E04 unknown_or_failed {'correlation': False, 'state': True, 'version': True}
E05 unknown_or_failed {'correlation': True, 'state': False, 'version': False}
E06 verified {'correlation': True, 'state': True, 'version': True}
文本误报数: 2


In [5]:
retry_without_readback = '重试取消' if events[2]['reply'] == '超时' else '不重试'  # 模拟超时后不读权威状态的危险策略。
fixed_status, fixed_evidence = verify(events[2])  # 先回读订单状态再判断同一超时事件。
print('失败案例 E03：盲目策略=', retry_without_readback, '，权威回读=', fixed_status, fixed_evidence)  # 展示超时不等于未生效。
print('生产差距：真实系统需关联 ID、事务/事件版本、最终一致轮询窗口、审计日志和 pending 人工升级队列。')  # 说明教学验证器的生产替换点。

失败案例 E03：盲目策略= 重试取消 ，权威回读= verified_after_readback {'correlation': True, 'state': True, 'version': True}
生产差距：真实系统需关联 ID、事务/事件版本、最终一致轮询窗口、审计日志和 pending 人工升级队列。


In [6]:
assert verify(events[0])[0] == 'verified'  # 验证关联、状态和版本全部正确时才确认完成。
assert verify(events[1])[0] == 'unknown_or_failed'  # 验证文本成功但权威状态错误不会误报完成。
assert fixed_status == 'verified_after_readback'  # 验证超时后读取权威状态能避免重复执行。